# 01_05 - KDE4 Data Collection

## Mục tiêu

Notebook này thực hiện bước **Data Collection** cho source:

**KDE4**

Nguồn:

**OPUS KDE4 v2**

Các nhiệm vụ:

1. Kiểm tra môi trường
2. Xác định project root
3. Cấu hình source
4. Download dataset
5. Convert dữ liệu sang DataFrame
6. Inspect raw data
7. Thống kê raw data
8. Xác định technology candidate
9. Tổng hợp audit summary
10. Lưu raw JSONL
11. Lưu raw Parquet
12. Lưu audit summary
13. Lưu metadata
14. Final verification
15. Ghi trạng thái notebook

> Lưu ý:
> - Notebook này không cleaning dữ liệu
> - Không deduplication
> - Không train/validation/test split
> - Không overwrite raw data
> - Technology candidate không đồng nghĩa với usable IT corpus
> - `usable_count` chưa được xác định
> - License chưa được xác nhận trong notebook này
> - Raw source files và provenance artifacts phải được giữ nguyên

In [1]:
import sys
import os
from pathlib import Path

print("Python:", sys.version)
print("Working directory:", os.getcwd())

Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
Working directory: C:\Users\ADMIN\ENVI-IT-MT\notebooks\01_data_collection


In [2]:
import datasets
import pandas as pd
import httpx

print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("httpx:", httpx.__version__)

datasets: 5.0.1
pandas: 3.0.5
httpx: 0.28.1


In [3]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path) -> Path:
    """
    Tìm project root bằng cách kiểm tra:
    - data/
    - notebooks/ hoặc notebook/
    """
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (
            (path / "data").is_dir()
            and (
                (path / "notebooks").is_dir()
                or (path / "notebook").is_dir()
            )
        ):
            return path

    raise FileNotFoundError(
        "Không tìm thấy project root. "
        "Hãy kiểm tra lại vị trí notebook."
    )


PROJECT_ROOT = find_project_root(CURRENT_DIR)

print("Project root:")
print(PROJECT_ROOT)

Project root:
C:\Users\ADMIN\ENVI-IT-MT


In [4]:
SOURCE_NAME = "KDE4"
SOURCE_SHORT_NAME = "kde4"

SOURCE_URL = "https://opus.nlpl.eu/KDE4"
DOWNLOAD_URL = "https://object.pouta.csc.fi/OPUS-KDE4/v2/moses/en-vi.txt.zip"

DATASET_ID = "OPUS-kde4-v2-eng-vie"
LANGUAGE_PAIR = "en-vi"
DOMAIN = "Software Localization"
DATASET_VERSION = "v2"

DOWNLOAD_METHOD = "OPUS direct download - Moses ZIP"

COLLECTION_DATE = pd.Timestamp.now().strftime("%Y-%m-%d")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / SOURCE_SHORT_NAME
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Source:", SOURCE_NAME)
print("Source URL:", SOURCE_URL)
print("Download URL:", DOWNLOAD_URL)
print("Dataset ID:", DATASET_ID)
print("Language pair:", LANGUAGE_PAIR)
print("Domain:", DOMAIN)
print("Dataset version:", DATASET_VERSION)
print("Download method:", DOWNLOAD_METHOD)
print("Raw directory:", RAW_DIR)

Source: KDE4
Source URL: https://opus.nlpl.eu/KDE4
Download URL: https://object.pouta.csc.fi/OPUS-KDE4/v2/moses/en-vi.txt.zip
Dataset ID: OPUS-kde4-v2-eng-vie
Language pair: en-vi
Domain: Software Localization
Dataset version: v2
Download method: OPUS direct download - Moses ZIP
Raw directory: C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4


In [5]:
import io
import zipfile
import httpx

# ==================================================
# Cell 6 - Download
# ==================================================

DATASET_ID = "OPUS-kde4-v2-eng-vie"

DOWNLOAD_URL = (
    "https://object.pouta.csc.fi/"
    "OPUS-KDE4/v2/moses/en-vi.txt.zip"
)

TRAIN_PARTS_DIR = (
    RAW_DIR
    / "train-parts"
)

SOURCE_FILE = (
    TRAIN_PARTS_DIR
    / f"{DATASET_ID}.eng"
)

TARGET_FILE = (
    TRAIN_PARTS_DIR
    / f"{DATASET_ID}.vie"
)

ZIP_PATH = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_opus.zip"
)

LICENSE_FILE = (
    RAW_DIR
    / "LICENSE"
)

README_FILE = (
    RAW_DIR
    / "README"
)

XML_FILE = (
    RAW_DIR
    / f"KDE4.en-vi.xml"
)

# --------------------------------------------------
# Kiểm tra dataset đã được download hoàn chỉnh chưa
# --------------------------------------------------

dataset_already_downloaded = (
    ZIP_PATH.is_file()
    and SOURCE_FILE.is_file()
    and TARGET_FILE.is_file()
    and LICENSE_FILE.is_file()
    and README_FILE.is_file()
    and XML_FILE.is_file()
)

if dataset_already_downloaded:

    print("Dataset đã tồn tại.")
    print("Bỏ qua bước download lại.")

    print("\nEnglish source:")
    print(SOURCE_FILE)

    print("\nVietnamese target:")
    print(TARGET_FILE)

    print("\nLicense:")
    print(LICENSE_FILE)

    print("\nREADME:")
    print(README_FILE)

    print("\nXML:")
    print(XML_FILE)

else:

    print("Dataset chưa hoàn chỉnh.")
    print("Download trực tiếp từ OPUS...")

    response = httpx.get(
        DOWNLOAD_URL,
        follow_redirects=True,
        timeout=120
    )

    print("\nHTTP status:", response.status_code)

    response.raise_for_status()

    ZIP_PATH.write_bytes(
        response.content
    )

    print("Downloaded:")
    print(ZIP_PATH)

    print(
        "Size (bytes):",
        ZIP_PATH.stat().st_size
    )

    # --------------------------------------------------
    # Extract archive
    # --------------------------------------------------

    TRAIN_PARTS_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        io.BytesIO(response.content),
        "r"
    ) as z:

        members = z.namelist()

        print("\nFiles inside archive:")

        for member in members:
            print(member)

        # --------------------------------------------------
        # Xác định English / Vietnamese files
        # --------------------------------------------------

        source_member = next(
            (
                member
                for member in members
                if member.endswith(".en")
            ),
            None
        )

        target_member = next(
            (
                member
                for member in members
                if member.endswith(".vi")
            ),
            None
        )

        if source_member is None:
            raise FileNotFoundError(
                "Không tìm thấy file English (.en) "
                "trong archive."
            )

        if target_member is None:
            raise FileNotFoundError(
                "Không tìm thấy file Vietnamese (.vi) "
                "trong archive."
            )

        # --------------------------------------------------
        # Save English
        # --------------------------------------------------

        with z.open(source_member) as src_file:
            SOURCE_FILE.write_bytes(
                src_file.read()
            )

        # --------------------------------------------------
        # Save Vietnamese
        # --------------------------------------------------

        with z.open(target_member) as tgt_file:
            TARGET_FILE.write_bytes(
                tgt_file.read()
            )

        # --------------------------------------------------
        # Save LICENSE
        # --------------------------------------------------

        if "LICENSE" in members:

            with z.open("LICENSE") as license_file:
                LICENSE_FILE.write_bytes(
                    license_file.read()
                )

        else:

            raise FileNotFoundError(
                "Archive không chứa LICENSE."
            )

        # --------------------------------------------------
        # Save README
        # --------------------------------------------------

        if "README" in members:

            with z.open("README") as readme_file:
                README_FILE.write_bytes(
                    readme_file.read()
                )

        else:

            raise FileNotFoundError(
                "Archive không chứa README."
            )

        # --------------------------------------------------
        # Save original XML
        # --------------------------------------------------

        xml_member = next(
            (
                member
                for member in members
                if member.endswith(".xml")
            ),
            None
        )

        if xml_member is None:
            raise FileNotFoundError(
                "Archive không chứa file XML."
            )

        with z.open(xml_member) as xml_file:
            XML_FILE.write_bytes(
                xml_file.read()
            )

    print("\nExtracted:")

    print(SOURCE_FILE)
    print(TARGET_FILE)
    print(LICENSE_FILE)
    print(README_FILE)
    print(XML_FILE)

# --------------------------------------------------
# Final Cell 6 verification
# --------------------------------------------------

required_files = [
    SOURCE_FILE,
    TARGET_FILE,
    LICENSE_FILE,
    README_FILE,
    XML_FILE
]

print("\nCell 6 verification:\n")

for file_path in required_files:

    print(
        f"{file_path.name:50}"
        f"{'OK' if file_path.is_file() else 'MISSING'}"
    )

if not all(
    file_path.is_file()
    for file_path in required_files
):
    raise FileNotFoundError(
        "Một hoặc nhiều raw/provenance files "
        "chưa được tạo đầy đủ."
    )

print("\nCell 6 completed successfully.")
print("Dataset ID:", DATASET_ID)
print("Output directory:", RAW_DIR)

Dataset đã tồn tại.
Bỏ qua bước download lại.

English source:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\train-parts\OPUS-kde4-v2-eng-vie.eng

Vietnamese target:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\train-parts\OPUS-kde4-v2-eng-vie.vie

License:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\LICENSE

README:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\README

XML:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\KDE4.en-vi.xml

Cell 6 verification:

OPUS-kde4-v2-eng-vie.eng                          OK
OPUS-kde4-v2-eng-vie.vie                          OK
LICENSE                                           OK
README                                            OK
KDE4.en-vi.xml                                    OK

Cell 6 completed successfully.
Dataset ID: OPUS-kde4-v2-eng-vie
Output directory: C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4


In [6]:
with open(
    SOURCE_FILE,
    "r",
    encoding="utf-8"
) as f:
    en_lines = [
        line.rstrip("\n")
        for line in f
    ]

with open(
    TARGET_FILE,
    "r",
    encoding="utf-8"
) as f:
    vi_lines = [
        line.rstrip("\n")
        for line in f
    ]

if len(en_lines) != len(vi_lines):
    raise ValueError(
        "English và Vietnamese không có cùng số dòng."
    )

df = pd.DataFrame(
    {
        "en": en_lines,
        "vi": vi_lines
    }
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Shape: (42782, 2)

Columns:
['en', 'vi']

Data types:
en    str
vi    str
dtype: object


In [7]:
print("First 5 rows:")
display(df.head())

print("\nRandom 5 rows:")
display(
    df.sample(
        5,
        random_state=42
    )
)

First 5 rows:


,en,vi
0,Add Feed to Akregator,Thêm nguồn tin cho Akregator
1,Add Feeds to Akregator,Thêm các nguồn tin cho Akregator
2,Add All Found Feeds to Akregator,Thêm mọi nguồn tin cho Akregator
3,Subscribe to site updates (using news feed),Theo dõi chỗ Mạng này tìm bản cập nhật (dùng n...
4,Imported Feeds,Nguồn tin đã nhập



Random 5 rows:


,en,vi
32178,Save All,Lưu & tất cả
13144,Get new color schemes from the Internet,Bộ màu
25994,Pes,Ghi chú
31275,New & Group...,Tên
40232,7.0,7. 0


In [8]:
raw_count = len(df)

missing_values = df.isna().sum()

duplicate_count = df.duplicated().sum()

print("Raw count:", raw_count)

print("\nMissing values:")
display(missing_values)

print("\nDuplicate rows:")
print(duplicate_count)

print("\nColumn statistics:")

for column in df.columns:
    print(f"\n{column}")
    print(
        "Non-null:",
        df[column].notna().sum()
    )
    print(
        "Unique:",
        df[column].nunique(
            dropna=False
        )
    )

Raw count: 42782

Missing values:


en    0
vi    0
dtype: int64


Duplicate rows:
2888

Column statistics:

en
Non-null: 42782
Unique: 35401

vi
Non-null: 42782
Unique: 29550


In [9]:
df_tech_candidate = df.copy()

technology_candidate_count = len(
    df_tech_candidate
)

non_technology_count = (
    raw_count
    - technology_candidate_count
)

print("Raw rows:", raw_count)

print(
    "Technology candidate rows:",
    technology_candidate_count
)

print(
    "Non-technology rows:",
    non_technology_count
)

Raw rows: 42782
Technology candidate rows: 42782
Non-technology rows: 0


In [10]:
audit_summary = {
    "source": SOURCE_NAME,
    "dataset_id": DATASET_ID,
    "dataset_version": DATASET_VERSION,
    "raw_count": int(raw_count),
    "candidate_count": int(technology_candidate_count),
    "non_technology_count": int(non_technology_count),
    "missing_values": {
        str(key): int(value)
        for key, value in missing_values.items()
    },
    "duplicate_count": int(duplicate_count),
    "columns": [str(column) for column in df.columns],
    "language_pair": LANGUAGE_PAIR,
    "source_segments": int(raw_count),
    "download_errors": 0,
    "candidate_rule": (
        "All rows from the KDE4 English-Vietnamese software "
        "localization corpus are retained as technology candidates. "
        "This does not establish final usable IT status."
    ),
}

audit_summary

{'source': 'KDE4',
 'dataset_id': 'OPUS-kde4-v2-eng-vie',
 'dataset_version': 'v2',
 'raw_count': 42782,
 'candidate_count': 42782,
 'non_technology_count': 0,
 'missing_values': {'en': 0, 'vi': 0},
 'duplicate_count': 2888,
 'columns': ['en', 'vi'],
 'language_pair': 'en-vi',
 'source_segments': 42782,
 'download_errors': 0,
 'candidate_rule': 'All rows from the KDE4 English-Vietnamese software localization corpus are retained as technology candidates. This does not establish final usable IT status.'}

In [11]:
raw_jsonl_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.jsonl"
)

df.to_json(
    raw_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved:")
print(raw_jsonl_path)

Saved:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\kde4_raw.jsonl


In [12]:
raw_parquet_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.parquet"
)

df.to_parquet(
    raw_parquet_path,
    index=False
)

print("Saved:")
print(raw_parquet_path)

Saved:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\kde4_raw.parquet


In [13]:
import json

audit_path = (
    RAW_DIR
    / "audit_summary.json"
)

with open(
    audit_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:")
print(audit_path)

Saved:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\audit_summary.json


In [14]:
metadata = {
    "source": SOURCE_NAME,
    "source_url": SOURCE_URL,
    "download_url": DOWNLOAD_URL,
    "license": "source-dependent",
    "version_revision": DATASET_VERSION,
    "collection_date": COLLECTION_DATE,
    "download_method": DOWNLOAD_METHOD,
    "language_pair": LANGUAGE_PAIR,
    "domain": DOMAIN,
    "subcategory": "KDE software localization",
    "raw_count": int(raw_count),
    "candidate_count": int(technology_candidate_count),
    "usable_count": None,
    "notes": (
        "Raw English-Vietnamese KDE4 localization corpus collected "
        "from OPUS KDE4 v2. The archive license applies according "
        "to the original sources; each original-source license must "
        "be checked before final use. Technology candidate count "
        "does not establish final usable IT status. Language check, "
        "alignment check, cleaning, deduplication, noise assessment "
        "and IT subdomain validation have not yet been completed."
    ),
}

metadata_path = RAW_DIR / "metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved:")
print(metadata_path)

Saved:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\metadata.json


In [15]:
expected_files = [
    ZIP_PATH,
    SOURCE_FILE,
    TARGET_FILE,
    LICENSE_FILE,
    README_FILE,
    XML_FILE,
    raw_jsonl_path,
    raw_parquet_path,
    audit_path,
    metadata_path,
]

verification_results = {
    file_path.name: file_path.is_file()
    for file_path in expected_files
}

print("Final verification:\n")

for file_name, exists in verification_results.items():
    print(f"{file_name:55}{'OK' if exists else 'MISSING'}")

verification_passed = all(verification_results.values())

if not verification_passed:
    missing_files = [
        name for name, exists in verification_results.items()
        if not exists
    ]
    raise FileNotFoundError(
        f"KDE4 thiếu file bắt buộc: {missing_files}"
    )

print("\nVerification passed:", verification_passed)

Final verification:

kde4_opus.zip                                          OK
OPUS-kde4-v2-eng-vie.eng                               OK
OPUS-kde4-v2-eng-vie.vie                               OK
LICENSE                                                OK
README                                                 OK
KDE4.en-vi.xml                                         OK
kde4_raw.jsonl                                         OK
kde4_raw.parquet                                       OK
audit_summary.json                                     OK
metadata.json                                          OK

Verification passed: True


# Data Collection Status

Source:

**KDE4 — OPUS v2**

| Metric | Value |
|---|---:|
| Raw rows | Recorded after download |
| Technology candidate rows | Same as raw count |
| Non-technology rows | 0 |
| Usable IT rows | TBD |

## Completed in this notebook

- [x] Environment checked
- [x] Project root identified
- [x] Source configured
- [x] KDE4 English-Vietnamese corpus downloaded
- [x] Raw parallel files extracted
- [x] Raw schema inspected
- [x] Raw statistics recorded
- [x] Technology candidate identified
- [x] Raw JSONL saved
- [x] Raw Parquet saved
- [x] Audit summary saved
- [x] Metadata saved
- [x] LICENSE preserved
- [x] README preserved
- [x] Original XML preserved
- [x] Raw source files verified
- [x] Provenance files verified
- [x] Output files verified

## Not completed in this notebook

- [ ] Full language check
- [ ] Alignment check
- [ ] Data cleaning
- [ ] Deduplication
- [ ] Quality/noise assessment
- [ ] IT subdomain classification
- [ ] Final usable IT count
- [ ] Train / validation / test split
- [ ] Original-source license verification

## Interpretation

KDE4 is a software-localization corpus and is therefore retained
as a technology candidate at the source-collection stage.

The archive LICENSE states:

> The data set comes with the same license
> as the original sources.

Therefore, the complete KDE4 dataset does not have one single
license assigned by this notebook.

Original-source license information must be checked before
the dataset is used in the final artifact.

However:

`technology_candidate_count != usable_count`

The final usable IT corpus must only be determined after the
subsequent audit, cleaning and IT-filtering stages.

> Raw data under `data/raw/kde4/` must remain unchanged.
>
> This notebook does not produce the final IT corpus.